# M3 분석 스튜디오 예시 — 계약 경유 조회 → 층별 → 관리도

노트북 규칙(계획서 M3): **원본(스테이징) 직접 접근 금지** — 표준 데이터셋(계약에 있는 테이블)만 조회한다.
이 노트북은 현업 리더의 30분 교육 시나리오 3개를 그대로 따라간다.

> 사전 조건: `python -m demo.run_e2e` 실행으로 `demo/out/axp.db`가 만들어져 있을 것.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'core'))  # demo/ 에서 실행 기준
from axp.custody import contracts

# 시나리오 0 — 내가 조회해도 되는 테이블은? 계약 목록이 곧 허가 목록이다
for name, doc in contracts.load_all().items():
    print(f"{name:24s} {doc['update_cycle']:9s} 소유자: {doc['owner']} — {doc['description'][:40]}")

In [ ]:
# 시나리오 1 — 층별: 어느 설비·근무조에서 불량이 많은가 (지난 여름)
from axp.studio import eda

t_eq, png_eq = eda.stratify('equipment_id', '2025-06-01', '2025-08-31')
t_sh, png_sh = eda.stratify('shift', '2025-06-01', '2025-08-31')
display(t_eq)
from IPython.display import Image
Image(png_eq)

OVEN-2가 눈에 띄면 다음 질문은 "언제부터?" — 관리도가 답한다.

In [ ]:
# 시나리오 2 — 관리도: 불량률이 관리 상태를 벗어난 날들
t_cc, png_cc, ooc_days = eda.control_chart('2025-06-01', '2025-08-31', equipment_id='OVEN-2')
print('관리 이탈일:', ooc_days[:10], '…' if len(ooc_days) > 10 else '')
Image(png_cc)

In [ ]:
# 시나리오 3 — 그래서 왜? 그래프에게 물어본다 (M6 조립기 — 전 문장 근거 인용)
from axp.judge import assembler
print(assembler.answer('OVEN-2 불량의 원인 후보는?', assembler.search_cause('OVEN-2')))

여기서 멈춘다 — 원인의 **단정**은 노트북의 일이 아니다. 후보의 확신도가 임계(70%·3회)에
도달하면 지식 에이전트가 승격을 상신하고, 사람이 승인해야 Rule이 된다.
노트북은 질문을 날카롭게 만드는 곳이지, 원장을 바꾸는 곳이 아니다.